## Imports

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math
import os
from tqdm import tqdm
import copy


## Building blocks
for the base transformer, including rotary embeddings, RMS normalization, SwiGLU activation

In [30]:
# ============================================================================
# Model Architecture
# ============================================================================

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization"""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class RotaryEmbedding(nn.Module):
    """Rotary Position Embedding (RoPE)"""
    def __init__(self, dim, max_seq_len=512):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos_cached', emb.cos())
        self.register_buffer('sin_cached', emb.sin())

    def forward(self, x):
        seq_len = x.shape[1]
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    # Original cos/sin shape: [seq_len, head_dim]
    # q/k shape: [batch_size, n_heads, seq_len, head_dim]
    # We need cos/sin to be [1, 1, seq_len, head_dim] for proper broadcasting
    cos = cos.unsqueeze(0).unsqueeze(1)  # Corrected from unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(1)  # Corrected from unsqueeze(2)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class SwiGLU(nn.Module):
    """SwiGLU activation function"""
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with RoPE"""
    def __init__(self, dim, n_heads, max_seq_len=512):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, max_seq_len)

        # Causal mask
        mask = torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(x)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.mask[:T, :T], float('-inf'))
        att = F.softmax(att, dim=-1)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class TransformerBlock(nn.Module):
    """Single transformer block with pre-norm"""
    def __init__(self, dim, n_heads, mlp_ratio=4, max_seq_len=512):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = CausalSelfAttention(dim, n_heads, max_seq_len)
        self.norm2 = RMSNorm(dim)
        self.mlp = SwiGLU(dim, dim * mlp_ratio)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


## The Tiny Recursive Model

In [31]:

class TinyRecursiveNetwork(nn.Module):
    """
    The core tiny network used in TRM.
    Only 2 layers as per the paper's finding that smaller is better.
    """
    def __init__(self, dim, n_heads=8, n_layers=2, mlp_ratio=4, max_seq_len=512):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlock(dim, n_heads, mlp_ratio, max_seq_len)
            for _ in range(n_layers)
        ])
        self.norm = RMSNorm(dim)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)


class TinyRecursiveModel(nn.Module):
    """
    Tiny Recursive Model for Text Generation

    Architecture based on TRM paper:
    - Single tiny 2-layer network
    - Recursive reasoning with latent z and prediction y
    - Deep supervision across multiple improvement steps

    For text generation:
    - x: embedded input sequence (context)
    - y: current token predictions (embedded)
    - z: latent reasoning state

    The model recursively improves its latent z, then updates y.
    """
    def __init__(
        self,
        vocab_size,
        dim=256,
        n_heads=8,
        n_layers=2,
        mlp_ratio=4,
        max_seq_len=256,
        n_latent_recursions=6,  # n in the paper
        n_improvement_cycles=3,  # T in the paper
    ):
        super().__init__()
        self.dim = dim
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.n_latent_recursions = n_latent_recursions
        self.n_improvement_cycles = n_improvement_cycles

        # Embeddings
        self.token_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)

        # Single tiny network (key insight: one network is better than two)
        self.net = TinyRecursiveNetwork(dim, n_heads, n_layers, mlp_ratio, max_seq_len)

        # Projection layers for combining x, y, z
        self.combine_xyz = nn.Linear(dim * 3, dim, bias=False)
        self.combine_yz = nn.Linear(dim * 2, dim, bias=False)

        # Output head
        self.output_head = nn.Linear(dim, vocab_size, bias=False)

        # Halting head for ACT (simplified - no Q-learning)
        self.halt_head = nn.Linear(dim, 1, bias=False)

        # Learnable initial states for y and z
        self.y_init = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.z_init = nn.Parameter(torch.randn(1, 1, dim) * 0.02)

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def get_embeddings(self, input_ids):
        """Get token + position embeddings"""
        B, T = input_ids.shape
        # Clamp input_ids to valid range
        input_ids = input_ids.clamp(0, self.vocab_size - 1)
        # Clamp position to max_seq_len
        T = min(T, self.max_seq_len)
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        return self.token_emb(input_ids[:, :T]) + self.pos_emb(pos)

    def latent_recursion(self, x, y, z):
        """
        Single recursion cycle:
        1. Update z n times given (x, y, z)
        2. Update y once given (y, z)
        """
        # Latent reasoning: update z n times
        for _ in range(self.n_latent_recursions):
            combined = self.combine_xyz(torch.cat([x, y, z], dim=-1))
            z = self.net(combined)

        # Refine prediction: update y given (y, z)
        combined_yz = self.combine_yz(torch.cat([y, z], dim=-1))
        y = self.net(combined_yz)

        return y, z

    def deep_recursion(self, x, y, z, use_grad=True):
        """
        Deep recursion with T improvement cycles.
        First T-1 cycles without gradients, last cycle with gradients.
        """
        if not use_grad:
            # All cycles without gradients (inference)
            with torch.no_grad():
                for _ in range(self.n_improvement_cycles):
                    y, z = self.latent_recursion(x, y, z)
            return y.detach(), z.detach()

        # T-1 cycles without gradients
        with torch.no_grad():
            for _ in range(self.n_improvement_cycles - 1):
                y, z = self.latent_recursion(x, y, z)

        # Last cycle with gradients
        y, z = self.latent_recursion(x, y, z)

        return y.detach(), z.detach(), self.output_head(y), self.halt_head(y.mean(dim=1))

    def forward(self, input_ids, targets=None, n_supervision_steps=4):
        """
        Forward pass with deep supervision.

        Args:
            input_ids: [B, T] input token IDs
            targets: [B, T] target token IDs (for training)
            n_supervision_steps: number of deep supervision steps

        Returns:
            If training: loss
            If inference: logits
        """
        B, T = input_ids.shape
        T = min(T, self.max_seq_len)
        input_ids = input_ids[:, :T]

        x = self.get_embeddings(input_ids)

        # Initialize y and z
        y = self.y_init.expand(B, T, -1).clone()
        z = self.z_init.expand(B, T, -1).clone()

        if targets is None:
            # Inference: just run deep recursion
            y, z = self.deep_recursion(x, y, z, use_grad=False)
            return self.output_head(y)

        # Ensure targets match input length
        targets = targets[:, :T]

        # Training with deep supervision
        total_loss = 0.0

        for step in range(n_supervision_steps):
            y, z, logits, halt_logit = self.deep_recursion(x, y, z, use_grad=True)

            # Cross-entropy loss for token prediction
            ce_loss = F.cross_entropy(
                logits.view(-1, self.vocab_size),
                targets.reshape(-1),
                ignore_index=-100
            )

            # Halting loss (simplified ACT)
            with torch.no_grad():
                preds = logits.argmax(dim=-1)
                mask = (targets != -100)
                correct = ((preds == targets) & mask).float().sum() / mask.float().sum().clamp(min=1)
            halt_loss = F.binary_cross_entropy_with_logits(
                halt_logit.squeeze(-1),
                correct.expand(B)
            )

            total_loss = total_loss + ce_loss + 0.1 * halt_loss

        return total_loss / n_supervision_steps

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=0.8, top_k=40):
        """Generate text autoregressively"""
        self.eval()

        for _ in range(max_new_tokens):
            # Crop to max_seq_len - 1 to leave room for prediction
            idx_cond = input_ids[:, -(self.max_seq_len - 1):]

            # Clamp input ids to valid vocab range
            idx_cond = idx_cond.clamp(0, self.vocab_size - 1)

            # Get predictions
            logits = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            # Top-k sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)

        return input_ids



In [32]:

# # ============================================================================
# # Dataset
# # ============================================================================

# class TinyStoriesDataset(Dataset):
#     """Dataset for TinyStories"""
#     def __init__(self, tokenizer, split='train', max_length=256, max_samples=None):
#         print(f"Loading TinyStories {split} split...")
#         dataset = load_dataset('roneneldan/TinyStories', split=split)

#         if max_samples:
#             dataset = dataset.select(range(min(max_samples, len(dataset))))

#         self.tokenizer = tokenizer
#         self.max_length = max_length
#         self.texts = dataset['text']
#         self.vocab_size = tokenizer.vocab_size
#         self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
#         print(f"Loaded {len(self.texts)} samples")

#     def __len__(self):
#         return len(self.texts)

#     def __getitem__(self, idx):
#         text = self.texts[idx]
#         # Add BOS/EOS handling
#         tokens = self.tokenizer.encode(text, truncation=True, max_length=self.max_length)

#         # Ensure all tokens are within valid range
#         tokens = [min(max(t, 0), self.vocab_size - 1) for t in tokens]

#         # Pad if necessary
#         if len(tokens) < self.max_length:
#             tokens = tokens + [self.pad_token_id] * (self.max_length - len(tokens))
#         else:
#             tokens = tokens[:self.max_length]

#         tokens = torch.tensor(tokens, dtype=torch.long)

#         # Input is tokens[:-1], target is tokens[1:]
#         input_ids = tokens[:-1].clone()
#         targets = tokens[1:].clone()

#         # Mask padding in targets (set to -100 to ignore in loss)
#         targets[targets == self.pad_token_id] = -100

#         return input_ids, targets


In [33]:


class ConvFinQADataset(Dataset):
    """Dataset cho ConvFinQA (Hỏi đáp Tài chính) sử dụng nguồn FinGPT"""
    def __init__(self, tokenizer, split='train', max_length=512, max_samples=None):
        print(f"Đang tải ConvFinQA split {split}...")

        # Đã đổi sang nguồn dữ liệu ổn định từ FinGPT
        dataset = load_dataset('FinGPT/fingpt-convfinqa', split=split)

        if max_samples:
            dataset = dataset.select(range(min(max_samples, len(dataset))))

        self.tokenizer = tokenizer
        self.max_length = max_length
        self.vocab_size = tokenizer.vocab_size
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

        # Xử lý format dạng QA
        self.texts = []
        for item in dataset:
            # FinGPT sử dụng format: instruction, input, output
            instruction = item.get('instruction', '')
            input_text = item.get('input', '')

            prompt = f"{instruction}\n{input_text}".strip()
            # Đề phòng cấu trúc dữ liệu bị đổi
            if not prompt:
                prompt = item.get('question', '')

            answer = item.get('output', '') or item.get('answer', '')

            # Format thành dạng Prompt - Response
            formatted_text = f"### Question:\n{prompt}\n\n### Answer:\n{answer}"
            self.texts.append(formatted_text)

        print(f"Đã tải {len(self.texts)} mẫu dữ liệu")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        tokens = self.tokenizer.encode(text, truncation=True, max_length=self.max_length)
        tokens = [min(max(t, 0), self.vocab_size - 1) for t in tokens]

        if len(tokens) < self.max_length:
            tokens = tokens + [self.pad_token_id] * (self.max_length - len(tokens))
        else:
            tokens = tokens[:self.max_length]

        tokens = torch.tensor(tokens, dtype=torch.long)

        input_ids = tokens[:-1].clone()
        targets = tokens[1:].clone()

        targets[targets == self.pad_token_id] = -100

        return input_ids, targets

## Exponential Moving Average
for weight regularization

In [34]:
class EMA:
    """Exponential Moving Average for model weights"""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}

        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (
                    self.decay * self.shadow[name] +
                    (1 - self.decay) * param.data
                )

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]


## Main Training Function

In [35]:
def train(
    model,
    train_loader,
    val_loader,
    tokenizer,
    device,
    epochs=5,
    lr=1e-4,
    warmup_steps=1000,
    n_supervision_steps=4,
    ema_decay=0.999,
    save_path='trm_convfinqa.pt'
):
    """Training loop with deep supervision and EMA"""

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    ema = EMA(model, decay=ema_decay)

    # Learning rate scheduler with warmup
    def lr_schedule(step):
        if step < warmup_steps:
            return step / warmup_steps
        return 1.0

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)

    global_step = 0
    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')

        for input_ids, targets in pbar:
            input_ids = input_ids.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            loss = model(input_ids, targets, n_supervision_steps=n_supervision_steps)
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()
            ema.update()

            global_step += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.6f}'})

        # Validation
        ema.apply_shadow()
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for input_ids, targets in tqdm(val_loader, desc='Validation'):
                input_ids = input_ids.to(device)
                targets = targets.to(device)
                loss = model(input_ids, targets, n_supervision_steps=n_supervision_steps)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        print(f'Epoch {epoch+1} - Val Loss: {val_loss:.4f}')

        # Generate sample
        # prompt = "Once upon a time"
        # prompt_ids = torch.tensor([tokenizer.encode(prompt)], device=device)
        # generated = model.generate(prompt_ids, max_new_tokens=100)

        # Generate sample (Dùng format của ConvFinQA)
        prompt = "### Question: What was the total revenue for the third quarter?\n### Answer:"
        # Có thể tăng max_new_tokens lên một chút nếu câu trả lời tài chính thường dài
        prompt_ids = torch.tensor([tokenizer.encode(prompt)], device=device)
        generated = model.generate(prompt_ids, max_new_tokens=150) # Tăng từ 100 lên 150

        generated_text = tokenizer.decode(generated[0].tolist())
        print(f'Generated: {generated_text[:300]}...\n')

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'ema_shadow': ema.shadow,
                'epoch': epoch,
                'val_loss': val_loss
            }, save_path)
            print(f'Saved best model with val_loss={val_loss:.4f}')

        ema.restore()

    return model

## Configuration

In [36]:
# # ============================================================================
# # Main
# # ============================================================================

# #def main():
# # Configuration
# config = {
#     'vocab_size': 50257,  # GPT-2 vocab
#     'dim': 256,           # Hidden dimension
#     'n_heads': 8,         # Attention heads
#     'n_layers': 2,        # Only 2 layers (key insight from paper)
#     'mlp_ratio': 4,
#     'max_seq_len': 512,   # Reduced for stability
#     'n_latent_recursions': 4,  # n in paper (reduced for memory)
#     'n_improvement_cycles': 2,  # T in paper (reduced for memory)

#     # Training
#     'batch_size': 4,     # 16 Reduced batch size
#     'epochs': 3,
#     'lr': 1e-4,
#     'warmup_steps': 500,
#     'n_supervision_steps': 3,  # Deep supervision steps during training
#     'max_train_samples': 2000000,  # Limit for faster training demo
#     'max_val_samples': 20000,
# }

config = {
    'vocab_size': 50257,  # GPT-2 vocab (Giữ nguyên nếu bạn dùng tokenizer của GPT-2)
    'dim': 256,
    'n_heads': 8,
    'n_layers': 2,
    'mlp_ratio': 4,
    'max_seq_len': 512,   # Đã tăng: Bắt buộc cho dữ liệu tài chính (QA)
    'n_latent_recursions': 4,
    'n_improvement_cycles': 2,

    # Training
    'batch_size': 4,      # Đã giảm: Để tránh OOM khi max_seq_len tăng
    'epochs': 3,          # Có thể tăng lên 5-10 vì dataset ConvFinQA nhỏ hơn TinyStories rất nhiều
    'lr': 1e-4,
    'warmup_steps': 500,
    'n_supervision_steps': 3,
    'max_train_samples': None, # Đã đổi: Lấy toàn bộ ~30k mẫu của ConvFinQA
    'max_val_samples': None,   # Đã đổi: Lấy toàn bộ tập val
}

## Instantiate Model



In [37]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Model
model = TinyRecursiveModel(
    vocab_size=config['vocab_size'],
    dim=config['dim'],
    n_heads=config['n_heads'],
    n_layers=config['n_layers'],
    mlp_ratio=config['mlp_ratio'],
    max_seq_len=config['max_seq_len'],
    n_latent_recursions=config['n_latent_recursions'],
    n_improvement_cycles=config['n_improvement_cycles'],
)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,} ({n_params/1e6:.2f}M)')
print(f'Effective depth per supervision step: {config["n_improvement_cycles"] * (config["n_latent_recursions"] + 1) * config["n_layers"]}')

Using device: cuda
Model parameters: 28,289,536 (28.29M)
Effective depth per supervision step: 20


## Load dataset

In [38]:
# # # Datasets
# # train_dataset = TinyStoriesDataset(
# #     tokenizer,
# #     split='train',
# #     max_length=config['max_seq_len'] + 1,  # +1 for next token prediction
# #     max_samples=config['max_train_samples']
# # )
# # val_dataset = TinyStoriesDataset(
# #     tokenizer,
# #     split='validation',
# #     max_length=config['max_seq_len'] + 1,
# #     max_samples=config['max_val_samples']
# # )

# # train_loader = DataLoader(
# #     train_dataset,
# #     batch_size=config['batch_size'],
# #     shuffle=True,
# #     num_workers=2,
# #     pin_memory=True
# # )
# # val_loader = DataLoader(
# #     val_dataset,
# #     batch_size=config['batch_size'],
# #     num_workers=2,
# #     pin_memory=True
# # )

# # ============================================================================
# # Datasets & DataLoaders
# # ============================================================================

# # Đã đổi thành ConvFinQADataset
# train_dataset = ConvFinQADataset(
#     tokenizer,
#     split='train',
#     max_length=config['max_seq_len'] + 1,  # +1 là rất chuẩn xác để dịch chênh lệch input/target
#     max_samples=config['max_train_samples']
# )

# val_dataset = ConvFinQADataset(
#     tokenizer,
#     split='validation', # Mẹo: Nếu HuggingFace báo lỗi không tìm thấy split này, hãy thử đổi thành 'test'
#     max_length=config['max_seq_len'] + 1,
#     max_samples=config['max_val_samples']
# )

# # DataLoader được giữ nguyên hoàn toàn
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=config['batch_size'],
#     shuffle=True,
#     num_workers=2,
#     pin_memory=True
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=config['batch_size'],
#     num_workers=2,
#     pin_memory=True
# )
# ============================================================================
# Datasets & DataLoaders
# ============================================================================

# Khởi tạo tập Train
train_dataset = ConvFinQADataset(
    tokenizer,
    split='train',
    max_length=config['max_seq_len'] + 1,
    max_samples=config['max_train_samples']
)

# Khởi tạo tập Test (Sử dụng thay cho tập Validation)
val_dataset = ConvFinQADataset(
    tokenizer,
    split='test',  # ĐÃ SỬA THÀNH 'test'
    max_length=config['max_seq_len'] + 1,
    max_samples=config['max_val_samples']
)

# DataLoader được giữ nguyên
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    num_workers=2,
    pin_memory=True
)

Đang tải ConvFinQA split train...
Đã tải 11104 mẫu dữ liệu
Đang tải ConvFinQA split test...
Đã tải 1490 mẫu dữ liệu


In [39]:
import torch
print(torch.cuda.get_arch_list())

['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


## Train

In [ ]:
# Train
model = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer,
    device=device,
    epochs=config['epochs'],
    lr=config['lr'],
    warmup_steps=config['warmup_steps'],
    n_supervision_steps=config['n_supervision_steps'],
    save_path='trm_convfinqa.pt'  # <--- Thêm mỗi dòng này để lưu đúng tên
)

print('\nTraining complete!')

Validation: 100%|██████████| 373/373 [01:55<00:00,  3.23it/s]


Epoch 1 - Val Loss: 3.3363
Generated: ### Question: What was the total revenue for the third quarter?
### Answer:P 500 earnings report carefully.Based on the question-answer history (if provided), answer the last question. The answer may require mathematical calculation based on the data provided.

stock market for the change in the fol...

Saved best model with val_loss=3.3363


Validation: 100%|██████████| 373/373 [01:56<00:00,  3.20it/s]


Epoch 2 - Val Loss: 2.7470
Generated: ### Question: What was the total revenue for the third quarter?
### Answer:P 500 earnings report carefully.Based on the question-answer history (if provided), answer the last question. The answer may require mathematical calculation based on the data provided.

zservability of our earnings per share...

Saved best model with val_loss=2.7470


Epoch 3/3:  69%|██████▉   | 1913/2776 [21:03<09:31,  1.51it/s, loss=2.4378, lr=0.000100]

## Generate

In [ ]:
# # Final generation examples
# model.eval()
# ema = EMA(model)

# prompts = [
#     "Once upon a time",
#     "The little girl",
#     "One day, a rabbit",
#     "Tom and his friend"
# ]

# print('\n=== Generated Stories ===\n')
# for prompt in prompts:
#     prompt_ids = torch.tensor([tokenizer.encode(prompt)], device=device)
#     generated = model.generate(prompt_ids, max_new_tokens=150, temperature=0.8)
#     text = tokenizer.decode(generated[0].tolist())
#     print(f'Prompt: "{prompt}"')
#     print(f'Story: {text}\n')
#     print('-' * 50 + '\n')


# #if __name__ == '__main__':
# #    main()
# ============================================================================
# Final generation examples
# ============================================================================

model.eval()

# Áp dụng trọng số EMA đã mượt hóa để generate text xịn nhất
ema.apply_shadow()

# Đổi toàn bộ prompt sang dạng câu hỏi tài chính và format đúng chuẩn
prompts = [
    "### Question:\nWhat was the total net revenue for the second quarter of 2021?\n\n### Answer:\n",
    "### Question:\nDid the operating expenses increase or decrease compared to the previous year?\n\n### Answer:\n",
    "### Question:\nBased on the financial statement, what is the gross profit margin?\n\n### Answer:\n",
    "### Question:\nHow much was the net income attributable to shareholders?\n\n### Answer:\n"
]

print('\n=== Generated Financial Answers ===\n')
for prompt in prompts:
    # Mã hóa prompt
    prompt_ids = torch.tensor([tokenizer.encode(prompt)], device=device)

    # Generate text: Giảm temperature xuống 0.2 để nó trả lời nghiêm túc, không bịa số liệu
    generated = model.generate(prompt_ids, max_new_tokens=150, temperature=0.2)

    # Giải mã và in kết quả
    text = tokenizer.decode(generated[0].tolist())

    # Cắt phần in ra cho đẹp
    print(text)
    print('-' * 70 + '\n')

# Trả lại trọng số gốc nếu mày muốn làm gì đó tiếp theo
ema.restore()

In [ ]:
import string
import re
import torch
from tqdm import tqdm

class ConvFinQAEvaluator:
    """Evaluator tính toán Exact Match (EM) cho ConvFinQA theo chuẩn AdaptLLM"""
    def __init__(self, tokenizer, device):
        self.tokenizer = tokenizer
        self.device = device

    def _normalize_answer(self, s: str) -> str:
        """Hàm chuẩn hóa text theo chuẩn SQuAD / AdaptLLM để tính EM chính xác"""
        def remove_articles(text):
            return re.sub(r'\b(a|an|the)\b', ' ', text)

        def white_space_fix(text):
            return ' '.join(text.split())

        def remove_punc(text):
            exclude = set(string.punctuation)
            return ''.join(ch for ch in text if ch not in exclude)

        def lower(text):
            return text.lower()

        return white_space_fix(remove_articles(remove_punc(lower(s))))

    def compute_em(self, prediction: str, ground_truth: str) -> float:
        """Tính Exact Match (1.0 nếu khớp hoàn toàn sau chuẩn hóa, ngược lại 0.0)"""
        # Tách lấy phần câu trả lời sau thẻ ### Answer: nếu mô hình sinh ra cả prompt
        if "### Answer:" in prediction:
            prediction = prediction.split("### Answer:")[-1]
        if "### Answer:" in ground_truth:
            ground_truth = ground_truth.split("### Answer:")[-1]

        norm_pred = self._normalize_answer(prediction)
        norm_gt = self._normalize_answer(ground_truth)

        return 1.0 if norm_pred == norm_gt else 0.0

    @torch.no_grad()
    def evaluate(self, model, dataloader, max_new_tokens=30):
        """Chạy qua dataloader để tính tổng điểm EM trên toàn bộ tập đánh giá"""
        model.eval()
        total_em = 0.0
        total_samples = 0

        # Lưu ý: Nên để batch_size=1 khi eval tự hồi quy để tránh lệch vị trí do padding
        for input_ids, targets in tqdm(dataloader, desc="Calculating ConvFinQA EM"):
            # Lấy batch size thực tế
            B = input_ids.size(0)

            for b in range(B):
                ids = input_ids[b:b+1].to(self.device)
                target = targets[b:b+1]

                # 1. Khôi phục text gốc để tìm đoạn Prompt gốc (bỏ phần câu trả lời của nhãn)
                full_text = self.tokenizer.decode(ids[0].tolist())

                # Cắt prompt tại điểm bắt đầu của câu trả lời để mớm cho model sinh tiếp
                if "### Answer:" in full_text:
                    prompt = full_text.split("### Answer:")[0] + "### Answer:\n"
                else:
                    prompt = full_text

                prompt_ids = torch.tensor([self.tokenizer.encode(prompt)], device=self.device)

                # 2. Cho mô hình sinh câu trả lời tự hồi quy
                generated = model.generate(prompt_ids, max_new_tokens=max_new_tokens, temperature=0.1)

                # Lấy phần text mô hình sinh thêm ra ngoài
                pred_text = self.tokenizer.decode(generated[0][prompt_ids.size(1):].tolist())

                # 3. Lấy nhãn Ground Truth thực tế (bỏ các token -100)
                clean_target_ids = [t for t in target[0].tolist() if t != -100]
                gt_text = self.tokenizer.decode(clean_target_ids)

                # 4. Tính toán EM
                em_score = self.compute_em(pred_text, gt_text)
                total_em += em_score
                total_samples += 1

        # Trả về điểm phần trăm (từ 0 đến 100) giống như log log của AdaptLLM
        final_score = (total_em / total_samples) * 100
        return {"ConvFinQA_em": final_score}

In [ ]:
# Validation Loss như cũ...
        # ...

        # Thêm đoạn này để tính EM Metric kiểu AdaptLLM
        ema.apply_shadow() # Đảm bảo dùng trọng số mượt của EMA để test
        evaluator = ConvFinQAEvaluator(tokenizer, device)

        # Tạo một loader riêng với batch_size=1 để sinh text chuẩn, không bị nhiễu pad_token
        eval_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
        scores = evaluator.evaluate(model, eval_loader, max_new_tokens=20)
        print(f"[INFO] - scores: {scores}")

        ema.restore()